### A5.4.6. Build Automatic Differentiation Engine

**Problem:**

Build a reverse-mode automatic differentiation engine that records operations on values and computes gradients via backpropagation through the recorded computation graph.

**Explanation:**

Reverse-mode autodiff records each operation during the forward pass, building a tape. During the backward pass, it walks the tape in reverse, applying the chain rule to propagate gradients from the output back to each input.

How to build it:

1. Create a `Value` class that stores a scalar number and its gradient (initially 0).
2. Each `Value` also stores a `backward` function and a list of child `Value` objects that produced it.
3. Override `__add__` and `__mul__`: create a new `Value` with the computed result, and attach a `backward` function that distributes the gradient to the inputs using the chain rule.
4. For add: both inputs get the upstream gradient unchanged. For mul: each input gets the upstream gradient times the other input's value.
5. To compute gradients: set the output's gradient to 1, topologically sort the graph, then call each node's `backward` in reverse order.

**Example:**

For `z = (a + b) * b` with `a=2, b=3`: `z=15`, `dz/da=3`, `dz/db=2+2*3=8`.

In [ ]:
class Value:
    def __init__(self, data, children=(), backward_fn=None):
        self.data = data
        self.grad = 0.0
        self.children = children
        self.backward_fn = backward_fn or (lambda: None)

    def __add__(self, other):
        out = Value(self.data + other.data, children=(self, other))

        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out.backward_fn = backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, children=(self, other))

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out.backward_fn = backward
        return out

    def backward(self):
        topo_order = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    build_topo(child)
                topo_order.append(node)

        build_topo(self)
        self.grad = 1.0
        reversed_order = reversed(topo_order)
        for node in reversed_order:
            node.backward_fn()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


a = Value(2.0)
b = Value(3.0)
c = a + b
z = c * b

print(f"z = (a + b) * b = {z.data}")

z.backward()

print(f"dz/da = {a.grad}")
print(f"dz/db = {b.grad}")

**References:**

📘 Baydin, A. G. et al. (2018). *Automatic Differentiation in Machine Learning: a Survey* — JMLR

---

[⬅️ Previous: Build Topological Sort Scheduler](./05_build_topological_sort_scheduler.ipynb) | [Next: Build Operator Fusion Pass ➡️](./07_build_operator_fusion_pass.ipynb)